## Optional: Live Database Connection (for spot checks only)
* This notebook's primary data source is the CSV exports in `data/`, loaded below. 
* This connection exists only for occasional verification against the live database.

In [3]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
user=os.getenv('user')
database=os.getenv('database')
host=os.getenv('host')
port=os.getenv('port')
password=os.getenv('password')
# format: postgresql:user:password@localhost:port/database
conn_eda=f"postgresql://{user}:{password}@{host}:{port}/{database}"
engine_eda=create_engine(conn_eda)
# Checking Connection
pd.read_sql("select count(*)as total_customers from customers;", con=engine_eda)

,total_customers
0,102016


## Dynamic CSV Loader Script
* This python script automates the process of reading multiple `CSV` files from a directory.
* Scans the `../data` directory to find all available CSV files.
* Strips the `.csv` extension to use as a clean key, and loads each file into a dictionary of pandas dataframes.

In [4]:
files=os.listdir("../data")
print(files)
dfs={}
# Loop through every file ound in folder
for file in files:
    name=file.replace(".csv","")
    dfs[name]=pd.read_csv(f"../data/{file}")
# Acces & view one dataset for confirmation
dfs['revenue by operator']


['call derop rate by tower.csv', 'churn label.csv', 'churn timing by operator.csv', 'cohort retention.csv', 'monthly revenue trend.csv', 'recharge frequency trend.csv', 'revenue by operator.csv']


,operator,total_revenue,operator_rank
0,Zong,4.528778e+08,1
1,Jazz,4.180964e+08,2
2,Ufone,2.417598e+08,3
3,Telenor,4.011211e+07,4


## Inspect: Revenue by Operator

* Total combined revenue (prepaid recharges + postpaid payments) per operator, ranked highest to lowest.
* Small aggregated table expect 4 rows, zero nulls, no cleaning needed.

In [5]:
dfs["revenue by operator"].info()
print(dfs["revenue by operator"].isnull().sum())
dfs["revenue by operator"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   operator       4 non-null      str    
 1   total_revenue  4 non-null      float64
 2   operator_rank  4 non-null      int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 248.0 bytes
operator         0
total_revenue    0
operator_rank    0
dtype: int64


,total_revenue,operator_rank
count,4.000000e+00,4.000000
mean,2.882115e+08,2.500000
std,1.894696e+08,1.290994
min,4.011211e+07,1.000000
25%,1.913479e+08,1.750000
50%,3.299281e+08,2.500000
75%,4.267918e+08,3.250000
max,4.528778e+08,4.000000


## Inspect: Churn Label

* Behavioral churn flag built independently from `customers.status`, using recharge/payment silence.
* Check row count matches total customers, confirm only valid Yes/No values exist, and check for nulls.

In [6]:
dfs["churn label"].info()
print(dfs["churn label"].isnull().sum())
dfs["churn label"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 102016 entries, 0 to 102015
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   customer_id  102016 non-null  str  
 1   churned      102016 non-null  str  
dtypes: str(2)
memory usage: 2.7 MB
customer_id    0
churned        0
dtype: int64


,customer_id,churned
count,102016,102016
unique,102016,2
top,CUST83515,No
freq,1,64606


## Inspect: Churn Timing by Operator

* Early vs late churn breakdown per operator (churned within first 6 months vs after).
* Confirm row count (4 operators × 2 labels = 8 rows), check percentage columns sum correctly per operator, and check for nulls.

In [7]:
dfs["churn timing by operator"].info()
print(dfs["churn timing by operator"].isnull().sum())
dfs["churn timing by operator"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   operator                 8 non-null      str    
 1   churned_label            8 non-null      str    
 2   total_churned_customers  8 non-null      int64  
 3   total_sum                8 non-null      float64
 4   percentage_of_customers  8 non-null      float64
dtypes: float64(2), int64(1), str(2)
memory usage: 528.0 bytes
operator                   0
churned_label              0
total_churned_customers    0
total_sum                  0
percentage_of_customers    0
dtype: int64


,total_churned_customers,total_sum,percentage_of_customers
count,8.000000,8.000000,8.0000
mean,5431.375000,10862.750000,50.0000
std,5484.663603,6351.625681,36.0403
min,287.000000,1703.000000,15.6700
25%,1488.000000,7664.750000,16.3450
50%,2618.500000,12342.500000,50.0000
75%,9248.000000,15540.500000,83.6550
max,14287.000000,17063.000000,84.3300


## Inspect: Monthly Revenue Trend

* Monthly revenue per operator with month-over-month growth rate (via `LAG()`).
* Expect some nulls in `pre_revenue`/`growth_rate` — each operator's first tracked month has no prior month to compare against.

In [8]:
dfs["monthly revenue trend"].info()
print(dfs["monthly revenue trend"].isnull().sum())
dfs["monthly revenue trend"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 268 entries, 0 to 267
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   operator       268 non-null    str    
 1   total_revenue  268 non-null    float64
 2   month          268 non-null    str    
 3   pre_revenue    264 non-null    float64
 4   growth_rate    264 non-null    float64
dtypes: float64(3), str(2)
memory usage: 13.7 KB
operator         0
total_revenue    0
month            0
pre_revenue      4
growth_rate      4
dtype: int64


,total_revenue,pre_revenue,growth_rate
count,2.680000e+02,2.640000e+02,264.000000
mean,4.301665e+06,4.301741e+06,9.252462
std,3.647325e+06,3.663611e+06,34.203204
min,4.050000e+03,4.050000e+03,-59.070000
25%,8.856771e+05,8.856771e+05,-1.432500
50%,3.752401e+06,3.714314e+06,4.960000
75%,7.171722e+06,7.177481e+06,11.370000
max,1.545830e+07,1.545830e+07,448.930000


## Inspect: Recharge Frequency Trend

* Compares each prepaid customer's recharge count in their first 3 months vs most recent 3 months, labeled increasing/decreasing/stable.
* Confirm row count matches prepaid customer count, check value distribution across the three labels, and check for nulls.

In [9]:
dfs["recharge frequency trend"].info()
print(dfs["recharge frequency trend"].isnull().sum())
dfs["recharge frequency trend"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 86764 entries, 0 to 86763
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   customer_id                 86764 non-null  str  
 1   recharge_in_first_3_months  86764 non-null  int64
 2   recharge_in_recent_month    86764 non-null  int64
 3   frequency_trend             86764 non-null  str  
dtypes: int64(2), str(2)
memory usage: 4.1 MB
customer_id                   0
recharge_in_first_3_months    0
recharge_in_recent_month      0
frequency_trend               0
dtype: int64


,recharge_in_first_3_months,recharge_in_recent_month
count,86764.000000,86764.000000
mean,4.069153,4.105677
std,5.332356,5.373733
min,1.000000,1.000000
25%,1.000000,1.000000
50%,2.000000,2.000000
75%,4.000000,4.000000
max,40.000000,40.000000


## Inspect: Call Drop Rate by Tower

* Drop rate percentage per tower, ranked worst to best (towers with 50+ calls only).
* Confirm row count (towers passing the 50-call minimum), check drop rate range is realistic (0-100%), and check for nulls.

In [10]:
dfs["call derop rate by tower"].info()
print(dfs["call derop rate by tower"].isnull().sum())
dfs["call derop rate by tower"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tower_id              500 non-null    str    
 1   total_calls           500 non-null    int64  
 2   drop_rate_percentage  500 non-null    float64
 3   tower_rank            500 non-null    int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 19.2 KB
tower_id                0
total_calls             0
drop_rate_percentage    0
tower_rank              0
dtype: int64


,total_calls,drop_rate_percentage,tower_rank
count,500.000000,500.000000,500.000000
mean,7325.228000,8.015100,67.330000
std,83.763385,0.327523,31.195118
min,7097.000000,7.020000,1.000000
25%,7268.000000,7.800000,45.750000
50%,7323.500000,8.015000,67.500000
75%,7382.250000,8.232500,89.000000
max,7560.000000,8.820000,139.000000


## Inspect: Cohort Retention

* Retention percentage at 3/6/12 months for each signup-month cohort.
* Watch for misleading 0% values in recent cohorts that haven't had enough time to reach a checkpoint yet — handled in the next section.

In [11]:
dfs["cohort retention"].info()
print(dfs["cohort retention"].isnull().sum())
dfs["cohort retention"].describe()

<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cohort_month  67 non-null     str    
 1   cohort_size   67 non-null     int64  
 2   retained_3m   67 non-null     float64
 3   retained_6m   67 non-null     float64
 4   retained_12m  67 non-null     float64
dtypes: float64(3), int64(1), str(1)
memory usage: 3.2 KB
cohort_month    0
cohort_size     0
retained_3m     0
retained_6m     0
retained_12m    0
dtype: int64


,cohort_size,retained_3m,retained_6m,retained_12m
count,67.000000,67.000000,67.000000,67.000000
mean,1522.626866,90.709104,83.433284,70.217313
std,132.314689,22.960436,29.360951,35.560816
min,571.000000,0.000000,0.000000,0.000000
25%,1506.500000,96.365000,87.185000,74.015000
50%,1543.000000,98.480000,96.310000,86.740000
75%,1578.000000,99.085000,97.945000,93.775000
max,1653.000000,99.480000,98.710000,96.400000


## Calculate Cohort Lifespan and Filter Incomplete Retention Columns
* Find the maximum date in the `cohort_month` column to establish a stable, data anchored baseline.
* Compute the total active months (`month_since_cohort`) for each subscriber group.
* Apply `.loc[]` to replace placeholder retention percentages with `NaN` for recent months where the 3-month, 6-month, or 12-month periods have not yet completed.

In [12]:
# Convert cohort_month to datetime objects
dfs["cohort retention"]["cohort_month"]=pd.to_datetime(dfs["cohort retention"]["cohort_month"])
# Confirm datatype
dfs["cohort retention"]["cohort_month"].dtype
# Calculate the difference from te maximum date of dataset and find the difference
max_date=dfs["cohort retention"]["cohort_month"].max()
months_since_cohort=(max_date-dfs["cohort retention"]["cohort_month"])
# Extract total days and convert to approximate months
days=months_since_cohort.dt.days
dfs["cohort retention"]["month_since_cohort"]=round(days/30,2)
# Use .loc[] to mask retention columns based on cohort age
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<12,"retained_12m"]=np.nan
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<6,"retained_6m"]=np.nan
dfs["cohort retention"].loc[dfs["cohort retention"]["month_since_cohort"]<3,"retained_3m"]=np.nan
dfs["cohort retention"].tail(10)

,cohort_month,cohort_size,retained_3m,retained_6m,retained_12m,month_since_cohort
57,2025-10-01,1611,89.45,73.56,NaN,9.10
58,2025-11-01,1562,85.72,70.87,NaN,8.07
59,2025-12-01,1601,82.01,58.21,NaN,7.07
60,2026-01-01,1538,76.85,9.10,NaN,6.03
61,2026-02-01,1382,75.25,NaN,NaN,5.00
62,2026-03-01,1568,65.75,NaN,NaN,4.07
63,2026-04-01,1541,15.12,NaN,NaN,3.03
64,2026-05-01,1560,NaN,NaN,NaN,2.03
65,2026-06-01,1578,NaN,NaN,NaN,1.00
66,2026-07-01,571,NaN,NaN,NaN,0.00
